**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Containers & Reproducibility

> ⚠️ **Draft — code not machine-verified.** Requires Docker (not installed on the authoring machine) not available at authoring time. An instructor should run each block before teaching. Remove this banner after that pass.

The final boss of "works on my machine": ship the machine. From pinned environments (which you should *already* be doing) to Docker images that make any workshop in this curriculum runnable, bit-for-bit, years from now.

## 1. Pre-requisites

[Intro to OS](../Intro_OS/Intro_OS.ipynb) §6 (shell, environment variables). Install [Docker Desktop](https://docs.docker.com/get-docker/) or `apt install docker.io`.

---
### 🕐 Session 1 of 2 — *Environments, Pinned* (~35 min)
**Goal:** exact-version environments with conda/uv; understand why 'latest' is a time bomb.
**Feeds into:** Session 2 (Docker).

---

💡 **Intuition.** Your code is one layer of a tower: Python version, package versions, system libraries, OS. "Works on my machine" means "my tower happens to align." Reproducibility is *writing the tower down*, in increasing order of completeness: `requirements.txt` (packages) → lockfiles (exact versions + hashes) → containers (everything above the kernel).

```bash
# Level 1: declare intent (loose — future installs may drift)
echo "numpy>=1.26
scipy
matplotlib" > requirements.txt

# Level 2: lock reality (exact versions, hash-checked — this is the reproducible one)
uv venv .venv && source .venv/bin/activate
uv pip install -r requirements.txt
uv pip freeze > requirements.lock          # commit BOTH files to git

# conda flavor: environment.yml + `conda env export --no-builds > environment.lock.yml`
```

The habit that matters: the lockfile is **generated, committed, and used for installs**
(`uv pip install -r requirements.lock`); `requirements.txt` records intent for humans.

---
### 🕐 Session 2 of 2 — *Docker: Ship the Machine* (~40 min)
**Goal:** containerize one of this curriculum's workshops end-to-end.
**Builds on:** Session 1.

---

💡 **Intuition.** A container is **not** a virtual machine — it's a normal process wearing blinders: namespaces hide the host's filesystem/processes/network ([OS workshop](../Intro_OS/Intro_OS.ipynb) concepts, weaponized), and the *image* is a frozen filesystem built layer-by-layer from a recipe. Same kernel, zero boot time, identical everywhere.

```dockerfile
# Dockerfile — containerize the Filter Design workshop
FROM python:3.12-slim

WORKDIR /workshop
COPY requirements.lock .
RUN pip install --no-cache-dir -r requirements.lock jupyter

COPY Intro_DSP/Filter_Design.ipynb .
EXPOSE 8888
CMD ["jupyter", "notebook", "--ip=0.0.0.0", "--no-browser", "--allow-root"]
```

```bash
docker build -t sps/filter-design .
docker run -p 8888:8888 sps/filter-design
# → open the printed URL: the workshop runs in a sealed, shippable environment

# the three commands that cover 90% of daily use:
docker ps                 # what's running
docker exec -it <id> bash # shell into a running container
docker images             # what's on disk (and eating it — see `docker system prune`)
```

**Layer-cache discipline:** order Dockerfile lines least-changing → most-changing (deps
before code), so editing the notebook doesn't reinstall NumPy. **Data stays outside** via
volumes (`-v $PWD/data:/workshop/data`) — images are for software, not datasets.

## 3. Conclusion

Declare intent, lock reality, and when it matters, ship the machine. A good capstone PR for this repo: a Dockerfile at the root that runs *any* workshop.

---
## Where next

- [Intro to OS](../Intro_OS/Intro_OS.ipynb) — the namespaces underneath the blinders.
- [Intro to Git](../Intro_Git/Intro_Git.ipynb) — version the recipe next to the code.